In [2]:
# ==========================================
# FILE 1: LINEAR SVM TRAINING (FIXED)
# Path: .../1.MachineLearning/train_svm.py
# ==========================================

import pandas as pd
import numpy as np
import re
import unicodedata
import time
import warnings
import os
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.svm import LinearSVC
from scipy.sparse import hstack, csr_matrix

import optuna
from tqdm.auto import tqdm
tqdm.pandas()

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# SETUP PATHS
# ==========================================
DATA_DIR = r"D:\nlp\project\CS221\Data\Clean"
MODEL_DIR = r"D:\nlp\project\CS221\Notebooks\Phase_3\1.MachineLearning\Models"
TRAIN_PATH = f"{DATA_DIR}/train_clean.csv"
VAL_PATH = f"{DATA_DIR}/val_clean.csv"
TEST_PATH = f"{DATA_DIR}/test_clean.csv"

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading data...")
start_load = time.time()

df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)
df_train_full = pd.concat([df_train, df_val], ignore_index=True)

le = LabelEncoder()
y_train_full = le.fit_transform(df_train_full['label'])
y_test = le.transform(df_test['label'])
class_names = le.classes_

print(f"Data loaded in {time.time() - start_load:.2f} seconds.")

# ==========================================
# 2. TEXT PREPROCESSING
# ==========================================
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.replace('<url>', ' special_token_url ')
    text = text.replace('<username>', ' special_token_user ')
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Preprocessing data...")
start_preprocess = time.time()

X_train_full_text = df_train_full['text'].progress_apply(preprocess_text)
X_test_text = df_test['text'].progress_apply(preprocess_text)

print(f"Preprocessing done in {time.time() - start_preprocess:.2f} seconds.")

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
print("Extracting features...")
start_feat = time.time()

tfidf_word_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=10000, sublinear_tf=True, min_df=2)
tfidf_char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=1000, sublinear_tf=True)

def get_features_adv(text_series, vec_word, vec_char, fit=False):
    if fit:
        tfidf_word = vec_word.fit_transform(text_series)
        tfidf_char = vec_char.fit_transform(text_series)
    else:
        tfidf_word = vec_word.transform(text_series)
        tfidf_char = vec_char.transform(text_series)
    
    word_counts = text_series.apply(lambda x: len(x.split())).values
    word_counts_sparse = csr_matrix(word_counts).T
    avg_word_len = text_series.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0).values
    avg_word_len_sparse = csr_matrix(avg_word_len).T
    
    return hstack([tfidf_word, tfidf_char, word_counts_sparse, avg_word_len_sparse])

X_train_features = get_features_adv(X_train_full_text, tfidf_word_vectorizer, tfidf_char_vectorizer, fit=True)
X_test_features = get_features_adv(X_test_text, tfidf_word_vectorizer, tfidf_char_vectorizer, fit=False)

if hasattr(X_train_features, "tocsr"): X_train_features = X_train_features.tocsr()
if hasattr(X_test_features, "tocsr"): X_test_features = X_test_features.tocsr()

print(f"Features extracted in {time.time() - start_feat:.2f} seconds. Shape: {X_train_features.shape}")

# ==========================================
# 4. OPTUNA TUNING
# ==========================================
def objective_svm(trial, X, y):
    params = {'C': trial.suggest_float('C', 1e-4, 100, log=True), 'class_weight': 'balanced', 'max_iter': 3000}
    model = LinearSVC(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

print("\n=========================")
print("TUNING LINEAR SVM (Optuna)")
print("=========================")

start_tune = time.time()
N_TRIALS_SVM = 15
study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(lambda t: objective_svm(t, X_train_features, y_train_full), n_trials=N_TRIALS_SVM, show_progress_bar=True)
tune_duration = time.time() - start_tune

print(f"\n[SVM DONE] Best F1: {study_svm.best_value:.4f}")
print(f"Tuning Time: {tune_duration:.2f} seconds ({tune_duration/60:.2f} minutes)")
print("Best params:", study_svm.best_params)

# ==========================================
# 5. TRAIN FINAL SVM
# ==========================================
print("\n=========================")
print("TRAINING FINAL SVM MODEL")
print("=========================")

start_train = time.time()
best_svm = LinearSVC(**study_svm.best_params, class_weight='balanced')
best_svm.fit(X_train_features, y_train_full)
train_duration = time.time() - start_train

print(f"Training Time: {train_duration:.2f} seconds ({train_duration/60:.2f} minutes)")

# ==========================================
# 6. EVALUATION (INFERENCE TIME)
# ==========================================
print("\n=========================")
print("FINAL EVALUATION")
print("=========================")

# Tính thời gian inference (predict)
start_inf = time.time()
y_pred_svm = best_svm.predict(X_test_features)
end_inf = time.time()

inf_time_total = end_inf - start_inf
# --- SỬA LỖI Ở ĐÂY: Dùng shape[0] thay vì len() ---
num_samples = X_test_features.shape[0] 
inf_time_per_sample_ms = (inf_time_total / num_samples) * 1000

f1_svm = f1_score(y_test, y_pred_svm, average='macro')

print(f"Test F1-Macro: {f1_svm * 100:.2f}%")
print(f"Training Time: {train_duration:.2f} seconds")
print(f"Inference Time: {inf_time_total:.4f} seconds")
print(f"Speed: {inf_time_per_sample_ms:.4f} ms/sample")

print("\nClassification Report (SVM):")
print(classification_report(y_test, y_pred_svm, target_names=class_names, digits=4))

# ==========================================
# 7. SAVE MODEL
# ==========================================
print("\nSaving model...")
joblib.dump(best_svm, f"{MODEL_DIR}/svm_model.pkl")
joblib.dump(tfidf_word_vectorizer, f"{MODEL_DIR}/tfidf_word.pkl")
joblib.dump(tfidf_char_vectorizer, f"{MODEL_DIR}/tfidf_char.pkl")
joblib.dump(le, f"{MODEL_DIR}/label_encoder.pkl")
print("Saved to:", MODEL_DIR)

total_time = time.time() - start_load
print(f"\nTOTAL EXECUTION TIME: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

Loading data...
Data loaded in 0.26 seconds.
Preprocessing data...


  0%|          | 0/41098 [00:00<?, ?it/s]

  0%|          | 0/7415 [00:00<?, ?it/s]

Preprocessing done in 2.32 seconds.
Extracting features...
Features extracted in 58.90 seconds. Shape: (41098, 11002)

TUNING LINEAR SVM (Optuna)


  0%|          | 0/15 [00:00<?, ?it/s]


[SVM DONE] Best F1: 0.7759
Tuning Time: 470.93 seconds (7.85 minutes)
Best params: {'C': 0.20506473147267987}

TRAINING FINAL SVM MODEL
Training Time: 61.89 seconds (1.03 minutes)

FINAL EVALUATION
Test F1-Macro: 77.91%
Training Time: 61.89 seconds
Inference Time: 0.0354 seconds
Speed: 0.0048 ms/sample

Classification Report (SVM):
              precision    recall  f1-score   support

     Anxiety     0.7658    0.8409    0.8016       836
  Depression     0.7476    0.6668    0.7049      2176
      Normal     0.9014    0.9434    0.9219      2723
    Suicidal     0.6829    0.6935    0.6881      1680

    accuracy                         0.7941      7415
   macro avg     0.7744    0.7862    0.7791      7415
weighted avg     0.7915    0.7941    0.7917      7415


Saving model...
Saved to: D:\nlp\project\CS221\Notebooks\Phase_3\1.MachineLearning\Models

TOTAL EXECUTION TIME: 594.95 seconds (9.92 minutes)
